In [1]:
import csv
import pandas as pd
import re

# Define path
input_path = "funding.csv"
output_path = "biodiversity_funding_supercleaned.csv"

# Function to clean each cell
def clean_text(text):
    if not isinstance(text, str):
        return text
    text = re.sub(r'\s+', ' ', text)  # Remove newline, tabs, multiple spaces
    text = re.sub(r'&#\d+;', '', text)  # Remove HTML unicode references like &#8203;
    text = re.sub(r'&nbsp;|&amp;|&lt;|&gt;', ' ', text)  # Replace common HTML entities
    text = re.sub(r'[\u200b\u200c\u200d\u200e\u200f]', '', text)  # Remove zero-width chars
    return text.strip()

# Step 1: Read file manually to fix bad rows
rows = []
with open(input_path, newline='', encoding='utf-8') as csvfile:
    reader = csv.reader(csvfile)
    for i, row in enumerate(reader):
        rows.append((i, row, len(row)))

# Get expected column count from header
header = rows[0][1]
expected_columns = len(header)

# Step 2: Clean and fix rows
cleaned_rows = []
for i, row, length in rows:
    if i == 0:
        cleaned_rows.append(row)
    elif length == expected_columns:
        cleaned_rows.append(row)
    elif i == 44 and length == 14:
        # Fix Row 45: merge columns 0 and 1
        fixed_row = [row[0] + row[1]] + row[2:]
        if len(fixed_row) == expected_columns:
            cleaned_rows.append(fixed_row)
    else:
        print(f"⚠️ Skipped row {i} with {length} columns: {row}")

# Step 3: Convert to DataFrame
df = pd.DataFrame(cleaned_rows[1:], columns=cleaned_rows[0])

# Step 4: Clean each cell
df_cleaned = df.applymap(clean_text)

# Step 5: Save cleaned CSV
df_cleaned.to_csv(output_path, index=False)
print(f"✅ Super-cleaned CSV saved to: {output_path}")

✅ Super-cleaned CSV saved to: biodiversity_funding_supercleaned.csv
